# 04 — Hierarchical Bayesian Model (Phase 3)
## The Core: Population Priors, Player Shrinkage, MCMC Inference

**Purpose:** Fit the full hierarchical Bayesian model that forms the backbone of Method 1.

**Model:**
```
Population level:   μ_i ~ N(μ_pop, τ²_pop)
Player level:       Y_{i,r,t} | μ_i ~ t(ν, μ_i, σ²_i)
```

**Key advantages over baseline:**
- Automatic shrinkage: sparse-data players pulled toward population mean
- Uncertainty quantification: posterior distributions, not point estimates
- Sub-component decomposition: separate OTT/APP/ARG/PUTT skill estimation
- Heavy-tailed noise: Student-t handles blowup rounds without distorting estimates

**Requirements:**
- `pymc>=5.10` installed (in environment.yml)
- Baseline model fitted (Notebook 03) for comparison


In [ ]:
# --- Setup ---
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config.settings import Settings
from data.loader import DataLoader
from features.pipeline import FeaturePipeline
from models.bayesian_core import HierarchicalBayesianModel
from models.priors import PriorConfig

cfg = Settings()
loader = DataLoader(cfg)
rounds_df = loader.load_rounds()

rounds_df = rounds_df.rename(columns={"year": "date", "dg_id": "player_id"})
rounds_df["date"] = pd.to_datetime(rounds_df["date"].astype(str) + "-01-01")

# Training data only
if "season" in rounds_df.columns:
    train_df = rounds_df[rounds_df["season"].isin(cfg.TRAIN_SEASONS)]
else:
    train_df = rounds_df

pipeline = FeaturePipeline(cfg)
train_features = pipeline.run(train_df)

print(f"Training data: {len(train_df):,} rounds, {train_df['player_id'].nunique():,} players")


In [ ]:
# --- Configure Priors ---
prior_config = PriorConfig()
print("Prior configuration:")
print(f"  Population mean (μ_pop):      N(0, 1)")
print(f"  Population spread (τ_pop):    HalfNormal(σ=1)")
print(f"  Player noise (σ_i):           HalfNormal(σ=2)")
print(f"  Observation d.f. (ν):         Exponential + 2 (ensures ν > 2)")
print(f"")
print(f"MCMC settings:")
print(f"  Draws:    {cfg.MCMC_DRAWS}")
print(f"  Tune:     {cfg.MCMC_TUNE}")
print(f"  Chains:   {cfg.MCMC_CHAINS}")
print(f"  Target accept: {cfg.MCMC_TARGET_ACCEPT}")


In [ ]:
# --- Fit the Bayesian Model ---
# ⚠️ This will take several minutes depending on your hardware.
# Start with ADVI (fast approximation) to verify the model works,
# then switch to MCMC for production estimates.

bayes_model = HierarchicalBayesianModel(cfg)

# Option A: Fast approximate inference (for development)
# bayes_model.fit(train_features.rounds_enriched, method="advi")

# Option B: Full MCMC (for production) — uncomment when ready:
bayes_model.fit(train_features.rounds_enriched, method="mcmc")

# print("\n⚠️ Uncomment the .fit() call above to run inference.")
# print("Recommended: start with ADVI, then switch to MCMC once confirmed working.")


In [ ]:
#--- Convergence Diagnostics (after fitting) ---
#Uncomment after fitting:

import arviz as az

trace = bayes_model.trace  # ArviZ InferenceData

# R-hat: should be < 1.01 for all parameters
rhat = az.rhat(trace)
print("R-hat summary:")
print(rhat)

# Effective sample size: should be > 400
ess = az.ess(trace)
print("\nEffective sample size:")
print(ess)

# Trace plots
az.plot_trace(trace, var_names=["mu_pop", "sigma_pop"])
plt.tight_layout()
plt.show()


In [ ]:
# --- Compare Posteriors: Top Players ---
# Uncomment after fitting:
#
from utils.plotting import plot_posterior_distributions

# Get top 10 players by posterior mean
posteriors_df = bayes_model.get_posteriors()
top_players = posteriors_df.nlargest(10, "mu_mean")

# Build posteriors dict using posterior samples from trace
posteriors = {}
for _, row in top_players.iterrows():
    pid = row["player_id"]
    idx = bayes_model.player_id_map[pid]
    samples = bayes_model.trace.posterior["mu_player"].values[:, :, idx].flatten()
    posteriors[f"Player {pid}"] = samples

fig, ax = plot_posterior_distributions(posteriors)
plt.show()


---
## ✅ Bayesian Model Fitted

**Convergence checklist:**
- [ ] All R-hat < 1.01
- [ ] All ESS > 400
- [ ] Trace plots show mixing (no stuck chains)
- [ ] Posterior means are reasonable (top players: SG ≈ 1–2, average: ≈ 0)

**If convergence fails:**
1. Try ADVI first to check model specification
2. Increase `MCMC_TUNE` to 2000
3. Reduce data to one season for debugging
4. Check for data issues (extreme SG values, wrong signs)

**Next step:** → **05_course_fit.ipynb**
